## Libraries

In [50]:
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity
import os
from langchain_community.document_loaders import ArxivLoader
from transformers import pipeline

## Wikipedia API Loader

In [51]:
CONFIG = {
    "topic": "Neural Network",
    "num_slides": 10,
    "top_k_documents": 20,
    "score_threshold": 0.0,
    "max_bullets_per_slide": 5
}

# Access like this:
topic = CONFIG["topic"]
num_slides = CONFIG["num_slides"]
top_k_documents = CONFIG["top_k_documents"]
score_threshold = CONFIG["score_threshold"]

# ═══════════════════════════════════════════════════════════════

In [52]:
def load_Wikidocs(topics):
    topics = str(topics)
    # Clean up special characters for better Wikipedia search
    clean_topics = topics.replace(".", "").replace("-", " ")
    loader = WikipediaLoader(query=clean_topics, load_max_docs=3)
    docs = loader.load()
    for doc in docs:
        doc.metadata['source'] = 'Wikipedia'
    return docs

In [53]:
def load_Arxivdocs(topics):
    topics = str(topics)
    
    try:
        loader = ArxivLoader(query=topics, load_max_docs=3)
        docs = loader.load()

        # If arXiv returns nothing
        if not docs:
            print("No arXiv docs found")
            return []

        # Add source tag
        for doc in docs:
            doc.metadata['source'] = 'arXiv'

        print(f"Loaded {len(docs)} docs from arXiv")
        return docs

    except Exception as e:
        print(f"arXiv FAILED: {e}")
        return []   # ← IMPORTANT (prevents crash)

In [54]:
arvix_docs = load_Arxivdocs(topic)

Loaded 3 docs from arXiv


In [55]:
wiki_docs = load_Wikidocs(topic)

In [56]:
print(len(arvix_docs))

3


In [57]:
arxiv_docs = load_Arxivdocs(topic)

Loaded 3 docs from arXiv


In [58]:
all_docs =  arxiv_docs + wiki_docs


## Chunking of Data


### Data -> Chunks

In [59]:
def split_documnets(documents,chunk_size = 1000, chunk_overlap = 200):
    """Splits documents into chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators= ["\n\n","\n"," ",""]
    )
    splits_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(splits_docs)} chunks")
    if splits_docs:
        print(f"\n Example chunk:")
        print(f"Content:{splits_docs[0].page_content}[:200]")
        print(f"Metadata:{splits_docs[0].metadata}")
    return splits_docs

In [60]:
chunks = split_documnets(all_docs)
chunks

Split 6 documents into 240 chunks

 Example chunk:
Content:A Tutorial about Random Neural Networks in Supervised
Learning
Sebastián Basterrech
Sebastian.Basterrech.Tiscordio@vsb.cz
Department of Computer Science
VŠB-Technical University of Ostrava
Ostrava-Poruba, Czech Republic
Gerardo Rubino
Gerardo.Rubino@inriafr
INRIA - Rennes
Rennes, France
Editor: –
Abstract
Random Neural Networks (RNNs) are a class of Neural Networks (NNs) that can also be
seen as a speciﬁc type of queuing network. They have been successfully used in several do-
mains during the last 25 years, as queuing networks to analyze the performance of resource
sharing in many engineering areas, as learning tools and in combinatorial optimization,
where they are seen as neural systems, and also as models of neurological aspects of living
beings. In this article we focus on their learning capabilities, and more speciﬁcally, we
present a practical guide for using the RNN to solve supervised learning problems. We give[:200]
M

[Document(metadata={'Published': '2016-09-15', 'Title': 'A Tutorial about Random Neural Networks in Supervised Learning', 'Authors': 'Sebastián Basterrech, Gerardo Rubino', 'Summary': 'Random Neural Networks (RNNs) are a class of Neural Networks (NNs) that can also be seen as a specific type of queuing network. They have been successfully used in several domains during the last 25 years, as queuing networks to analyze the performance of resource sharing in many engineering areas, as learning tools and in combinatorial optimization, where they are seen as neural systems, and also as models of neurological aspects of living beings. In this article we focus on their learning capabilities, and more specifically, we present a practical guide for using the RNN to solve supervised learning problems. We give a general description of these models using almost indistinctly the terminology of Queuing Theory and the neural one. We present the standard learning procedures used by RNNs, adapted from

## EMBEDDING 


### Text -> Vectors

In [61]:
class EmbeddingManager:
    """Handles document embedding using sentence Transformer"""
    #function to setup the environment
    
    def __init__(self,model_name:str = 'all-MiniLM-L6-v2'):
        """Initilizes the embedding manager
        model_name used in this case is a sentence tranformer model from Hugging Face whose default name is 'all-MiniLM-L6-v2'
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """Loads the Sentence Tranformer model"""
        try:
            print(f"Loading Embedding: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension:{self.model.get_sentence_embedding_dimension}")
        except Exception as e:
            print(f"Error loading model {self.model_name}{e}")
            raise

    def generate_embedding(self,text):
        """Generate embedding from list of text"""
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embedding for{len(text)}texts")
        embedding = self.model.encode(text,show_progress_bar= True)
        print(f"Generated Embedding with shape:{embedding.shape}")
        return embedding

    

In [62]:
embedding_manager= EmbeddingManager()
embedding_manager

Loading Embedding: all-MiniLM-L6-v2
Model loaded successfully. Embedding dimension:<bound method SentenceTransformer.get_sentence_embedding_dimension of SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)>


## Making of Vector DataBase

In [63]:
class VectorStore:
    
    def __init__(self, collection_name: str = "web_documents", persist_directory: str = "../data/vector/store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store() 
    def _initialize_store(self):
        """Initialaizes the ChromaDB client and collector"""\
        #Create the persistent ChromaDB client
        try:
            os.makedirs(self.persist_directory, exist_ok = True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #get or create collection
            self.collection = self.client.get_or_create_collection(
                name= self.collection_name,
                metadata={"description": "Web document embedding for RAG"}
            )
            print(f"Vector Store initialized. Collection{self.collection_name}")
            print(f"Existing document in collection:{self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store:{e}")
            raise
    
    def add_documents(self,documents:List[Any],embedding:np.ndarray):
        """Add documents and their embedding to vector store"""
        if len(documents) != len(embedding):
            raise ValueError("Numbers of documnets must match numbers of eembddding")
        print(f"Adding{len(documents)}documents to Vector Store")

        #Prepare for chromaDB
        ids = []
        metadatas = []
        documents_text = []
        embedding_list = []

        for i,(doc,embedding) in enumerate (zip(documents,embedding)):
            #Generate unique id's 
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #Prepare Metadata
            metadata = dict(doc.metadata)
            metadata['doc_index']= 1
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            #document content
            documents_text.append(doc.page_content)

            #embedding
            embedding_list.append(embedding.tolist())

        #add to collection
        try:
            self.collection.add(
                ids = ids,
                embeddings= embedding_list,
                metadatas= metadatas,
                documents= documents_text
            )
            print(f"Successfullr added {len(documents)}documents to Vector Store")
            print(f"total documents in collection:{self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to Vector Store:{e}")
            raise

In [64]:
# ─── Clear old data when topic changes ────────────────────────────────────────
import chromadb

client = chromadb.PersistentClient(path="../data/vector/store")  # use your actual path

# Delete old collections entirely
for name in ["arxiv_documents", "wiki_documents"]:
    try:
        client.delete_collection(name)
        print(f"Cleared collection: {name}")
    except:
        print(f"Collection {name} didn't exist, skipping")

Cleared collection: arxiv_documents
Cleared collection: wiki_documents


In [65]:
arxiv_vectorstore = VectorStore(collection_name="arxiv_documents")
wiki_vectorstore  = VectorStore(collection_name="wiki_documents")

Vector Store initialized. Collectionarxiv_documents
Existing document in collection:0
Vector Store initialized. Collectionwiki_documents
Existing document in collection:0


In [66]:
chunks = split_documnets(all_docs)

arxiv_chunks = [c for c in chunks if c.metadata.get('source') == 'arXiv']
wiki_chunks  = [c for c in chunks if c.metadata.get('source') == 'Wikipedia']

print("Total chunks:", len(chunks))
print("Unique sources:", set(c.metadata.get('source') for c in chunks))
print("arXiv chunks:", len(arxiv_chunks))
print("Wiki chunks:",  len(wiki_chunks))

Split 6 documents into 240 chunks

 Example chunk:
Content:A Tutorial about Random Neural Networks in Supervised
Learning
Sebastián Basterrech
Sebastian.Basterrech.Tiscordio@vsb.cz
Department of Computer Science
VŠB-Technical University of Ostrava
Ostrava-Poruba, Czech Republic
Gerardo Rubino
Gerardo.Rubino@inriafr
INRIA - Rennes
Rennes, France
Editor: –
Abstract
Random Neural Networks (RNNs) are a class of Neural Networks (NNs) that can also be
seen as a speciﬁc type of queuing network. They have been successfully used in several do-
mains during the last 25 years, as queuing networks to analyze the performance of resource
sharing in many engineering areas, as learning tools and in combinatorial optimization,
where they are seen as neural systems, and also as models of neurological aspects of living
beings. In this article we focus on their learning capabilities, and more speciﬁcally, we
present a practical guide for using the RNN to solve supervised learning problems. We give[:200]
M

In [67]:
if arxiv_chunks:
    arxiv_embeddings = embedding_manager.generate_embedding([d.page_content for d in arxiv_chunks])
    arxiv_vectorstore.add_documents(arxiv_chunks, arxiv_embeddings)
else:
    print("No arXiv chunks found, skipping")

if wiki_chunks:
    wiki_embeddings = embedding_manager.generate_embedding([d.page_content for d in wiki_chunks])
    wiki_vectorstore.add_documents(wiki_chunks, wiki_embeddings)
else:
    print("No Wiki chunks found, skipping")

Generating embedding for224texts


Batches: 100%|██████████| 7/7 [00:07<00:00,  1.07s/it]


Generated Embedding with shape:(224, 384)
Adding224documents to Vector Store
Successfullr added 224documents to Vector Store
total documents in collection:224
Generating embedding for16texts


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.34it/s]

Generated Embedding with shape:(16, 384)
Adding16documents to Vector Store
Successfullr added 16documents to Vector Store
total documents in collection:16


## RAG Pipeline for Vector Store

In [68]:
class RAGRetriever:
    def __init__(self, arxiv_store: VectorStore, wiki_store: VectorStore, embedding_manager: EmbeddingManager):
        self.arxiv_store = arxiv_store
        self.wiki_store  = wiki_store
        self.embedding_manager = embedding_manager

    def _query_store(self, collection, query_embedding, k: int) -> list:
        results = collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=k
        )
        docs = []
        if results['documents'] and results['documents'][0]:
            for doc_id, document, metadata, distance in zip(
                results['ids'][0], results['documents'][0],
                results['metadatas'][0], results['distances'][0]
            ):
                similarity_score = 1 / (1 + distance)
                docs.append({
                    'id': doc_id,
                    'content': document,
                    'metadata': metadata,
                    'similarity_score': similarity_score,
                    'distance': distance
                })
        return docs

    def retriever(self, query: str, top_k: int = top_k_documents,score_threshold: float = score_threshold, skip_arxiv: bool = False):
    
        print(f"Retrieving documents for query: '{query}'")
        k_each = top_k // 2

        query_embedding = self.embedding_manager.generate_embedding([query])[0]

        # Only query arxiv if it has data for this topic
        arxiv_docs = [] if skip_arxiv else self._query_store(self.arxiv_store.collection, query_embedding, k_each)
        wiki_docs  = self._query_store(self.wiki_store.collection, query_embedding, k_each)

        for doc in arxiv_docs:
            doc['weighted_score'] = doc['similarity_score'] * 0.5
        for doc in wiki_docs:
            doc['weighted_score'] = doc['similarity_score'] * 0.5

        combined = arxiv_docs + wiki_docs
        combined = [d for d in combined if d['weighted_score'] >= score_threshold]
        combined = sorted(combined, key=lambda x: x['weighted_score'], reverse=True)

        for i, doc in enumerate(combined):
            doc['rank'] = i + 1

        print(f"Retrieved {len(arxiv_docs)} arXiv + {len(wiki_docs)} Wikipedia docs")
        return combined

In [69]:

rag_retriever = RAGRetriever(arxiv_vectorstore, wiki_vectorstore, embedding_manager)
docs = rag_retriever.retriever(topic, skip_arxiv=(len(arxiv_chunks) == 0))

Retrieving documents for query: 'Neural Network'
Generating embedding for1texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 82.36it/s]

Generated Embedding with shape:(1, 384)
Retrieved 10 arXiv + 10 Wikipedia docs


## Summarization using Hugging Face (facebook/bart-large-cnn)


In [70]:
# ✅ FAST: Use a lighter model + batch processing (no OpenAI key needed)
from transformers import pipeline

# Use a smaller, faster model — great for students on CPU
summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-6-6",  # ~2x faster than bart-large-cnn, same quality
    device=-1  # CPU
)

# ✅ Batch all docs in one call instead of one-by-one loop
texts = [doc["content"] for doc in docs]

# Truncate long texts to avoid token limit errors
texts = [t[:800] for t in texts]

# Single batched call — much faster than looping
batch_summaries = summarizer(texts, max_length=60, min_length=20, do_sample=False, batch_size=4)
summaries = [s["summary_text"] for s in batch_summaries]

rag_text = " ".join(summaries)

In [71]:
# ─── STEP 1: Detect topic type ───────────────────────────────────────────────
def detect_topic_type(rag_text):
    academic_signals = ["neural", "algorithm", "equation", "research", "paper",
                        "proposed", "model", "training", "dataset", "published"]
    pop_signals = ["season", "episode", "character", "aired", "cast", "series",
                   "album", "band", "film", "award", "director", "starring"]
    text_lower = rag_text.lower()
    academic_score = sum(1 for w in academic_signals if w in text_lower)
    pop_score = sum(1 for w in pop_signals if w in text_lower)
    return "academic" if academic_score >= pop_score else "pop_culture"

topic_type = detect_topic_type(rag_text)
print(f"Detected topic type: {topic_type}")

# ─── STEP 2: Headings based on topic type ────────────────────────────────────
def generate_headings(topic, num_slides, topic_type):
    if topic_type == "academic":
        base = [topic, "Introduction", "History & Background", "How It Works",
                "Types & Categories", "Key Components", "Applications",
                "Advantages & Benefits", "Challenges & Limitations",
                "Future Scope", "Conclusion", "References"]
    else:
        base = [topic, "Introduction", "History & Background", "Key People",
                "Plot & Themes", "Reception", "Legacy & Impact",
                "Notable Episodes", "Cultural Impact",
                "Fun Facts", "Conclusion", "References"]
    return base[:num_slides]

# ─── STEP 3: Keywords based on topic type ────────────────────────────────────
if topic_type == "academic":
    keywords = {
        "Introduction":             ["overview", "definition", "what is", "refers to", "known as", "is a"],
        "History & Background":     ["history", "origin", "evolution", "introduced", "developed", "founded", "first", "1943", "1989"],
        "How It Works":             ["works", "process", "mechanism", "computes", "forward", "backpropagation", "weight", "activation"],
        "Types & Categories":       ["types", "kinds", "categories", "feedforward", "recurrent", "convolutional", "LSTM", "RNN", "CNN"],
        "Key Components":           ["neuron", "layer", "hidden", "input", "output", "bias", "weight", "node"],
        "Applications":             ["application", "used for", "applied", "image", "speech", "classification", "prediction", "recognition"],
        "Advantages & Benefits":    ["advantage", "benefit", "accurate", "faster", "efficient", "outperform", "better"],
        "Challenges & Limitations": ["challenge", "limitation", "problem", "drawback", "vanishing", "exploding", "difficult", "slow"],
        "Future Scope":             ["future", "research", "upcoming", "trend", "potential", "direction", "next"],
        "Conclusion":               ["summary", "conclude", "overall", "in summary", "this paper", "demonstrate"],
        "References":               []
    }
else:
    keywords = {
        "Introduction":             ["is a", "known as", "refers to", "american", "sitcom", "show", "television", "series", "film"],
        "History & Background":     ["created", "premiered", "first aired", "developed", "founded", "began", "started", "origin"],
        "Key People":               ["cast", "starring", "actor", "actress", "character", "played by", "creator", "writer", "director"],
        "Plot & Themes":            ["story", "plot", "theme", "episode", "season", "follows", "centers", "about"],
        "Reception":                ["award", "emmy", "rating", "critics", "audience", "popular", "acclaim", "won", "nominated"],
        "Legacy & Impact":          ["influence", "legacy", "iconic", "cultural", "impact", "inspired", "remembered"],
        "Notable Episodes":         ["episode", "season", "memorable", "famous", "notable", "best", "iconic scene"],
        "Cultural Impact":          ["culture", "reference", "meme", "phrase", "catchphrase", "phenomenon", "generation"],
        "Fun Facts":                ["fact", "trivia", "behind the scenes", "originally", "almost", "nearly", "first choice"],
        "Conclusion":               ["summary", "overall", "one of", "considered", "remains", "still"],
        "References":               []
    }

# ─── STEP 4: Build slides ─────────────────────────────────────────────────────
topic_words = set(topic.lower().split())
headings = generate_headings(topic, num_slides, topic_type)

final_slides = []
used_sentences = set()  # ← OUTSIDE the loop

for i, heading in enumerate(headings):
    if i == 0:
        final_slides.append({"title": topic, "bullets": []})
        continue

    # Query per slide
    slide_docs = rag_retriever.retriever(
        f"{topic} {heading}",
        top_k=20,
        skip_arxiv=(len(arxiv_chunks) == 0)
    )

    slide_sentences = []
    for d in slide_docs:
        # skip irrelevant arXiv docs for this topic
        if d["metadata"].get("source") == "arXiv":
            continue
        for sent in d["content"].replace(". ", ".|").split("|"):
            sent = sent.strip()
            if len(sent) > 20:
                slide_sentences.append(sent)

    # Score sentences
    scored = []
    for sent in slide_sentences:
        topic_score = sum(1 for w in topic_words if w in sent.lower())
        key_list = keywords.get(heading, [])
        keyword_score = sum(1 for kw in key_list if kw.lower() in sent.lower())
        total_score = topic_score + (keyword_score * 3)
        if total_score > 0:
            scored.append((total_score, sent))

    scored = sorted(scored, key=lambda x: x[0], reverse=True)

    # Deduplicate
    seen = set()
    bullets = []
    for _, sent in scored:
        norm = sent.strip().lower()
        if norm not in seen and norm not in used_sentences:
            seen.add(norm)
            used_sentences.add(norm)
            bullets.append(sent.strip())
        if len(bullets) == CONFIG["max_bullets_per_slide"]:
            break

    if not bullets:
        bullets = [s for s in slide_sentences if len(s) > 20][:CONFIG["max_bullets_per_slide"]]

    final_slides.append({"title": heading, "bullets": bullets})

Detected topic type: academic
Retrieving documents for query: 'Neural Network Introduction'
Generating embedding for1texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 55.51it/s]


Generated Embedding with shape:(1, 384)
Retrieved 10 arXiv + 10 Wikipedia docs
Retrieving documents for query: 'Neural Network History & Background'
Generating embedding for1texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 55.48it/s]


Generated Embedding with shape:(1, 384)
Retrieved 10 arXiv + 10 Wikipedia docs
Retrieving documents for query: 'Neural Network How It Works'
Generating embedding for1texts


...

Generated Embedding with shape:(1, 384)
Retrieved 10 arXiv + 10 Wikipedia docs
Retrieving documents for query: 'Neural Network Challenges & Limitations'
Generating embedding for1texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 65.87it/s]


Generated Embedding with shape:(1, 384)
Retrieved 10 arXiv + 10 Wikipedia docs
Retrieving documents for query: 'Neural Network Future Scope'
Generating embedding for1texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 71.42it/s]

Generated Embedding with shape:(1, 384)
Retrieved 10 arXiv + 10 Wikipedia docs


In [72]:
for i, slide in enumerate(final_slides):
    print(f"\nSlide {i+1}: {slide['title']}")
    for b in slide['bullets']:
        print(f"  - {b}")


Slide 1: Neural Network

Slide 2: Introduction
  - In machine learning, a neural network (NN) or neural net, also known as an artificial neural network (ANN), is a computational model inspired by the structure and functions of biological neural networks.
A neural network consists of connected units or nodes called artificial neurons, which loosely model the neurons in the brain.
  - In machine learning, a neural network is an artificial mathematical model used to approximate nonlinear functions.
  - While early artificial neural networks were physical machines, today they are almost always implemented in software.
Neurons in an artificial neural network are usually arranged into layers, with information passing from the first layer (the input layer) through one or more intermediate layers (the hidden layers) to the final layer (the output layer).
The "signal" input to each neuron is a number, specifically a linear combination of the outputs of the connected neurons in the previous lay